# Retail Intelligence Platform

# Notebook 02 - Data Cleaning

---

## Objective

This notebook prepares the raw Olist dataset for analysis.

Activities performed:

- Load raw datasets
- Create backup copies
- Standardize column names
- Convert data types
- Handle missing values
- Remove duplicate records
- Validate cleaned data
- Save cleaned datasets

---

## CRISP-DM Phase

✅ Business Understanding

✅ Data Understanding

✅ Data Preparation (Current)

⬜ Modeling

⬜ Evaluation

⬜ Deployment

---

## Output

Clean datasets stored in:

data/
    cleaned/

In [1]:
# ======================================================
# Notebook 02 - Data Cleaning
# ======================================================

import pandas as pd
import numpy as np

from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("="*70)
print("Retail Intelligence Platform")
print("Notebook 02 - Data Cleaning")
print("="*70)

Retail Intelligence Platform
Notebook 02 - Data Cleaning


In [2]:
# ======================================================
# Project Paths
# ======================================================

PROJECT_ROOT = Path.cwd().parent

RAW_DATA = PROJECT_ROOT / "data" / "raw"

CLEAN_DATA = PROJECT_ROOT / "data" / "cleaned"

CLEAN_DATA.mkdir(parents=True, exist_ok=True)

print("Raw Data :", RAW_DATA)

print("Cleaned Data :", CLEAN_DATA)

Raw Data : c:\Users\divya\Downloads\Retail-Intelligence-Platform\data\raw
Cleaned Data : c:\Users\divya\Downloads\Retail-Intelligence-Platform\data\cleaned


In [3]:
# ======================================================
# Load Raw Data
# ======================================================

datasets = {}

for file in sorted(RAW_DATA.glob("*.csv")):

    df = pd.read_csv(file)

    datasets[file.stem] = df

print("="*70)

print("Datasets Loaded")

print("="*70)

for name, df in datasets.items():

    print(f"{name:<45}{df.shape}")

Datasets Loaded
olist_customers_dataset                      (99441, 5)
olist_geolocation_dataset                    (1000163, 5)
olist_order_items_dataset                    (112650, 7)
olist_order_payments_dataset                 (103886, 5)
olist_order_reviews_dataset                  (99224, 7)
olist_orders_dataset                         (99441, 8)
olist_products_dataset                       (32951, 9)
olist_sellers_dataset                        (3095, 4)
product_category_name_translation            (71, 2)


In [4]:
# ======================================================
# Create Working Copies
# ======================================================

cleaned_data = {}

for name, df in datasets.items():

    cleaned_data[name] = df.copy(deep=True)

print(f"Working Copies Created : {len(cleaned_data)}")

Working Copies Created : 9


In [5]:
# ======================================================
# Cleaning Dashboard
# ======================================================

print("="*70)

print("DATA CLEANING DASHBOARD")

print("="*70)

print(f"Raw Datasets      : {len(datasets)}")

print(f"Working Copies    : {len(cleaned_data)}")

print("Cleaning Status   : Ready")

print("="*70)

DATA CLEANING DASHBOARD
Raw Datasets      : 9
Working Copies    : 9
Cleaning Status   : Ready


In [6]:
# ======================================================
# Automatic Data Type Conversion
# ======================================================

conversion_report = []

for dataset_name, df in cleaned_data.items():

    for column in df.columns:

        original_dtype = str(df[column].dtype)

        new_dtype = original_dtype

        conversion = "No Change"

        try:

            # Convert date columns
            if any(keyword in column.lower() for keyword in
                   ["date", "time", "timestamp"]):

                df[column] = pd.to_datetime(
                    df[column],
                    errors="coerce"
                )

                new_dtype = str(df[column].dtype)
                conversion = "Datetime"

            # Convert integer-like float columns
            elif str(df[column].dtype) == "float64":

                non_null = df[column].dropna()

                if len(non_null) > 0:

                    if (non_null % 1 == 0).all():

                        df[column] = df[column].astype("Int64")

                        new_dtype = str(df[column].dtype)
                        conversion = "Float → Int"

        except Exception as e:

            conversion = f"Error : {e}"

        conversion_report.append({

            "Dataset": dataset_name,
            "Column": column,
            "Original Type": original_dtype,
            "New Type": new_dtype,
            "Conversion": conversion

        })

conversion_df = pd.DataFrame(conversion_report)

display(conversion_df.head(20))

,Dataset,Column,Original Type,New Type,Conversion
0,olist_customers_dataset,customer_id,str,str,No Change
1,olist_customers_dataset,customer_unique_id,str,str,No Change
2,olist_customers_dataset,customer_zip_code_prefix,int64,int64,No Change
3,olist_customers_dataset,customer_city,str,str,No Change
4,olist_customers_dataset,customer_state,str,str,No Change
5,olist_geolocation_dataset,geolocation_zip_code_prefix,int64,int64,No Change
6,olist_geolocation_dataset,geolocation_lat,float64,float64,No Change
7,olist_geolocation_dataset,geolocation_lng,float64,float64,No Change
8,olist_geolocation_dataset,geolocation_city,str,str,No Change
9,olist_geolocation_dataset,geolocation_state,str,str,No Change


In [7]:
# ======================================================
# Data Type Conversion Summary
# ======================================================

print("="*70)
print("DATA TYPE CONVERSION SUMMARY")
print("="*70)

summary = (
    conversion_df
    .groupby("Conversion")
    .size()
    .reset_index(name="Columns")
)

display(summary)

print("\nTotal Columns Checked :", len(conversion_df))

DATA TYPE CONVERSION SUMMARY


,Conversion,Columns
0,Datetime,7
1,Float → Int,7
2,No Change,38



Total Columns Checked : 52


In [8]:
# ======================================================
# Verify Date Columns
# ======================================================

date_verification = []

for dataset_name, df in cleaned_data.items():

    for column in df.columns:

        if any(keyword in column.lower() for keyword in
               ["date", "time", "timestamp"]):

            date_verification.append({

                "Dataset": dataset_name,

                "Column": column,

                "Data Type": str(df[column].dtype),

                "Null Values": int(df[column].isnull().sum())

            })

date_df = pd.DataFrame(date_verification)

display(date_df)

,Dataset,Column,Data Type,Null Values
0,olist_order_items_dataset,shipping_limit_date,datetime64[us],0
1,olist_order_reviews_dataset,review_creation_date,datetime64[us],0
2,olist_order_reviews_dataset,review_answer_timestamp,datetime64[us],0
3,olist_orders_dataset,order_purchase_timestamp,datetime64[us],0
4,olist_orders_dataset,order_delivered_carrier_date,datetime64[us],1783
5,olist_orders_dataset,order_delivered_customer_date,datetime64[us],2965
6,olist_orders_dataset,order_estimated_delivery_date,datetime64[us],0


In [9]:
# ======================================================
# Memory Usage Report
# ======================================================

memory_report = []

for dataset_name, df in cleaned_data.items():

    memory_report.append({

        "Dataset": dataset_name,

        "Rows": len(df),

        "Columns": len(df.columns),

        "Memory (MB)": round(
            df.memory_usage(deep=True).sum()/1024**2,
            2
        )

    })

memory_df = pd.DataFrame(memory_report)

display(memory_df)

,Dataset,Rows,Columns,Memory (MB)
0,olist_customers_dataset,99441,5,11.03
1,olist_geolocation_dataset,1000163,5,50.12
2,olist_order_items_dataset,112650,7,16.33
3,olist_order_payments_dataset,103886,5,8.11
4,olist_order_reviews_dataset,99224,7,14.25
5,olist_orders_dataset,99441,8,14.80
6,olist_products_dataset,32951,9,3.95
7,olist_sellers_dataset,3095,4,0.22
8,product_category_name_translation,71,2,0.00


In [10]:
# ======================================================
# Cleaning Progress Dashboard
# ======================================================

print("="*70)
print("DATA CLEANING PROGRESS")
print("="*70)

print(f"Datasets Loaded      : {len(cleaned_data)}")
print(f"Columns Processed    : {len(conversion_df)}")

converted = (
    conversion_df["Conversion"] != "No Change"
).sum()

print(f"Columns Converted    : {converted}")

print(f"Date Columns Found   : {len(date_df)}")

print("="*70)

DATA CLEANING PROGRESS
Datasets Loaded      : 9
Columns Processed    : 52
Columns Converted    : 14
Date Columns Found   : 7


In [11]:
# ======================================================
# Missing Value Assessment
# ======================================================

missing_report = []

for dataset_name, df in cleaned_data.items():

    total_rows = len(df)

    for column in df.columns:

        missing = df[column].isnull().sum()

        missing_report.append({

            "Dataset": dataset_name,
            "Column": column,
            "Missing Values": missing,
            "Missing %": round(
                missing / total_rows * 100,
                2
            )

        })

missing_report_df = pd.DataFrame(missing_report)

missing_report_df = missing_report_df.sort_values(

    "Missing Values",

    ascending=False

)

display(missing_report_df.head(20))

,Dataset,Column,Missing Values,Missing %
25,olist_order_reviews_dataset,review_comment_title,87656,88.34
26,olist_order_reviews_dataset,review_comment_message,58247,58.70
35,olist_orders_dataset,order_delivered_customer_date,2965,2.98
34,olist_orders_dataset,order_delivered_carrier_date,1783,1.79
39,olist_products_dataset,product_name_lenght,610,1.85
38,olist_products_dataset,product_category_name,610,1.85
41,olist_products_dataset,product_photos_qty,610,1.85
40,olist_products_dataset,product_description_lenght,610,1.85
33,olist_orders_dataset,order_approved_at,160,0.16
42,olist_products_dataset,product_weight_g,2,0.01


In [12]:
# ======================================================
# Missing Value Classification
# ======================================================

def classify_missing(percent):

    if percent == 0:
        return "No Missing"

    elif percent < 5:
        return "Low"

    elif percent < 20:
        return "Medium"

    else:
        return "High"

missing_report_df["Severity"] = (

    missing_report_df["Missing %"]

    .apply(classify_missing)

)

display(missing_report_df)

,Dataset,Column,Missing Values,Missing %,Severity
25,olist_order_reviews_dataset,review_comment_title,87656,88.34,High
26,olist_order_reviews_dataset,review_comment_message,58247,58.70,High
35,olist_orders_dataset,order_delivered_customer_date,2965,2.98,Low
34,olist_orders_dataset,order_delivered_carrier_date,1783,1.79,Low
39,olist_products_dataset,product_name_lenght,610,1.85,Low
38,olist_products_dataset,product_category_name,610,1.85,Low
41,olist_products_dataset,product_photos_qty,610,1.85,Low
40,olist_products_dataset,product_description_lenght,610,1.85,Low
33,olist_orders_dataset,order_approved_at,160,0.16,Low
42,olist_products_dataset,product_weight_g,2,0.01,Low


In [13]:
# ======================================================
# Business Cleaning Strategy
# ======================================================

cleaning_strategy = []

for _, row in missing_report_df.iterrows():

    column = row["Column"]

    strategy = "Keep"

    if "review_comment_message" in column:

        strategy = "Keep Missing"

    elif "order_delivered_customer_date" in column:

        strategy = "Business Missing"

    elif "approved_at" in column:

        strategy = "Business Missing"

    elif row["Missing %"] > 50:

        strategy = "Review"

    cleaning_strategy.append({

        "Dataset": row["Dataset"],

        "Column": column,

        "Missing %": row["Missing %"],

        "Strategy": strategy

    })

strategy_df = pd.DataFrame(cleaning_strategy)

display(strategy_df)

,Dataset,Column,Missing %,Strategy
0,olist_order_reviews_dataset,review_comment_title,88.34,Review
1,olist_order_reviews_dataset,review_comment_message,58.70,Keep Missing
2,olist_orders_dataset,order_delivered_customer_date,2.98,Business Missing
3,olist_orders_dataset,order_delivered_carrier_date,1.79,Keep
4,olist_products_dataset,product_name_lenght,1.85,Keep
5,olist_products_dataset,product_category_name,1.85,Keep
6,olist_products_dataset,product_photos_qty,1.85,Keep
7,olist_products_dataset,product_description_lenght,1.85,Keep
8,olist_orders_dataset,order_approved_at,0.16,Business Missing
9,olist_products_dataset,product_weight_g,0.01,Keep


In [14]:
# ======================================================
# Missing Value Dashboard
# ======================================================

print("="*70)

print("MISSING VALUE DASHBOARD")

print("="*70)

print(

    f"Columns with Missing : "

    f"{(missing_report_df['Missing Values']>0).sum()}"

)

print(

    f"Total Missing Cells : "

    f"{missing_report_df['Missing Values'].sum():,}"

)

print("="*70)

display(

    strategy_df["Strategy"]

    .value_counts()

)

MISSING VALUE DASHBOARD
Columns with Missing : 13
Total Missing Cells : 153,259


Strategy
Keep                48
Business Missing     2
Review               1
Keep Missing         1
Name: count, dtype: int64

In [15]:
# ======================================================
# High Risk Columns
# ======================================================

high_risk = strategy_df[

    strategy_df["Missing %"] >= 20

]

display(high_risk)

,Dataset,Column,Missing %,Strategy
0,olist_order_reviews_dataset,review_comment_title,88.34,Review
1,olist_order_reviews_dataset,review_comment_message,58.70,Keep Missing


In [16]:
# ======================================================
# Cleaning Recommendation Report
# ======================================================

print("="*70)
print("CLEANING RECOMMENDATIONS")
print("="*70)

for _, row in high_risk.iterrows():

    print(

        f"{row['Dataset']} -> "

        f"{row['Column']} "

        f"({row['Missing %']}%) "

        f"Recommendation : {row['Strategy']}"

    )

print("="*70)

CLEANING RECOMMENDATIONS
olist_order_reviews_dataset -> review_comment_title (88.34%) Recommendation : Review
olist_order_reviews_dataset -> review_comment_message (58.7%) Recommendation : Keep Missing


In [17]:
# ======================================================
# Duplicate Cleaning Assessment
# ======================================================

duplicate_before = []

for dataset_name, df in cleaned_data.items():

    duplicate_before.append({

        "Dataset": dataset_name,

        "Rows Before": len(df),

        "Duplicate Rows": int(df.duplicated().sum())

    })

duplicate_before_df = pd.DataFrame(duplicate_before)

display(duplicate_before_df)

,Dataset,Rows Before,Duplicate Rows
0,olist_customers_dataset,99441,0
1,olist_geolocation_dataset,1000163,261831
2,olist_order_items_dataset,112650,0
3,olist_order_payments_dataset,103886,0
4,olist_order_reviews_dataset,99224,0
5,olist_orders_dataset,99441,0
6,olist_products_dataset,32951,0
7,olist_sellers_dataset,3095,0
8,product_category_name_translation,71,0


In [18]:
# ======================================================
# Remove Exact Duplicate Rows
# ======================================================

duplicate_log = []

for dataset_name, df in cleaned_data.items():

    before = len(df)

    df_clean = df.drop_duplicates().copy()

    after = len(df_clean)

    removed = before - after

    cleaned_data[dataset_name] = df_clean

    duplicate_log.append({

        "Dataset": dataset_name,

        "Rows Before": before,

        "Rows After": after,

        "Duplicates Removed": removed

    })

duplicate_log_df = pd.DataFrame(duplicate_log)

display(duplicate_log_df)

,Dataset,Rows Before,Rows After,Duplicates Removed
0,olist_customers_dataset,99441,99441,0
1,olist_geolocation_dataset,1000163,738332,261831
2,olist_order_items_dataset,112650,112650,0
3,olist_order_payments_dataset,103886,103886,0
4,olist_order_reviews_dataset,99224,99224,0
5,olist_orders_dataset,99441,99441,0
6,olist_products_dataset,32951,32951,0
7,olist_sellers_dataset,3095,3095,0
8,product_category_name_translation,71,71,0


In [19]:
# ======================================================
# Duplicate Removal Dashboard
# ======================================================

print("="*70)
print("DUPLICATE CLEANING DASHBOARD")
print("="*70)

total_removed = duplicate_log_df["Duplicates Removed"].sum()

print(f"Datasets Processed : {len(cleaned_data)}")
print(f"Total Rows Removed : {total_removed:,}")

print("="*70)

display(duplicate_log_df)

DUPLICATE CLEANING DASHBOARD
Datasets Processed : 9
Total Rows Removed : 261,831


,Dataset,Rows Before,Rows After,Duplicates Removed
0,olist_customers_dataset,99441,99441,0
1,olist_geolocation_dataset,1000163,738332,261831
2,olist_order_items_dataset,112650,112650,0
3,olist_order_payments_dataset,103886,103886,0
4,olist_order_reviews_dataset,99224,99224,0
5,olist_orders_dataset,99441,99441,0
6,olist_products_dataset,32951,32951,0
7,olist_sellers_dataset,3095,3095,0
8,product_category_name_translation,71,71,0


In [20]:
# ======================================================
# Duplicate Verification
# ======================================================

verification = []

for dataset_name, df in cleaned_data.items():

    verification.append({

        "Dataset": dataset_name,

        "Remaining Duplicates": int(df.duplicated().sum())

    })

verification_df = pd.DataFrame(verification)

display(verification_df)

,Dataset,Remaining Duplicates
0,olist_customers_dataset,0
1,olist_geolocation_dataset,0
2,olist_order_items_dataset,0
3,olist_order_payments_dataset,0
4,olist_order_reviews_dataset,0
5,olist_orders_dataset,0
6,olist_products_dataset,0
7,olist_sellers_dataset,0
8,product_category_name_translation,0


In [21]:
# ======================================================
# Cleaning Audit Report
# ======================================================

audit = duplicate_log_df.copy()

audit["Status"] = audit["Duplicates Removed"].apply(

    lambda x: "Cleaned" if x > 0 else "No Action"

)

display(audit)

,Dataset,Rows Before,Rows After,Duplicates Removed,Status
0,olist_customers_dataset,99441,99441,0,No Action
1,olist_geolocation_dataset,1000163,738332,261831,Cleaned
2,olist_order_items_dataset,112650,112650,0,No Action
3,olist_order_payments_dataset,103886,103886,0,No Action
4,olist_order_reviews_dataset,99224,99224,0,No Action
5,olist_orders_dataset,99441,99441,0,No Action
6,olist_products_dataset,32951,32951,0,No Action
7,olist_sellers_dataset,3095,3095,0,No Action
8,product_category_name_translation,71,71,0,No Action


In [22]:
# ======================================================
# Executive Cleaning Summary
# ======================================================

print("="*70)
print("EXECUTIVE CLEANING SUMMARY")
print("="*70)

datasets_cleaned = len(cleaned_data)

duplicates_removed = audit["Duplicates Removed"].sum()

datasets_modified = (audit["Duplicates Removed"] > 0).sum()

print(f"Datasets Cleaned      : {datasets_cleaned}")
print(f"Datasets Modified     : {datasets_modified}")
print(f"Duplicate Rows Removed: {duplicates_removed:,}")

print("="*70)

EXECUTIVE CLEANING SUMMARY
Datasets Cleaned      : 9
Datasets Modified     : 1
Duplicate Rows Removed: 261,831


In [23]:
# ======================================================
# Business Rule Validation Engine
# ======================================================

business_rules = []

# -----------------------------
# Orders Dataset
# -----------------------------

orders = cleaned_data["olist_orders_dataset"]

# Purchase Date <= Delivery Date

invalid_delivery = (

    orders["order_delivered_customer_date"]

    <

    orders["order_purchase_timestamp"]

).sum()

business_rules.append({

    "Dataset":"olist_orders_dataset",

    "Rule":"Delivery after Purchase",

    "Violations":int(invalid_delivery)

})

# Estimated Date >= Purchase Date

invalid_estimated = (

    orders["order_estimated_delivery_date"]

    <

    orders["order_purchase_timestamp"]

).sum()

business_rules.append({

    "Dataset":"olist_orders_dataset",

    "Rule":"Estimated Date after Purchase",

    "Violations":int(invalid_estimated)

})

rule_df = pd.DataFrame(business_rules)

display(rule_df)

,Dataset,Rule,Violations
0,olist_orders_dataset,Delivery after Purchase,0
1,olist_orders_dataset,Estimated Date after Purchase,0


In [24]:
# ======================================================
# Payment Validation
# ======================================================

payments = cleaned_data["olist_order_payments_dataset"]

payment_validation = pd.DataFrame({

    "Check":[

        "Negative Payments",

        "Zero Payments",

        "Negative Installments"

    ],

    "Violations":[

        (payments["payment_value"] < 0).sum(),

        (payments["payment_value"] == 0).sum(),

        (payments["payment_installments"] < 0).sum()

    ]

})

display(payment_validation)

,Check,Violations
0,Negative Payments,0
1,Zero Payments,9
2,Negative Installments,0


In [25]:
# ======================================================
# Product Validation
# ======================================================

products = cleaned_data["olist_products_dataset"]

product_validation = pd.DataFrame({

    "Check":[

        "Negative Weight",

        "Negative Length",

        "Negative Height",

        "Negative Width"

    ],

    "Violations":[

        (products["product_weight_g"] < 0).sum(),

        (products["product_length_cm"] < 0).sum(),

        (products["product_height_cm"] < 0).sum(),

        (products["product_width_cm"] < 0).sum()

    ]

})

display(product_validation)

,Check,Violations
0,Negative Weight,0
1,Negative Length,0
2,Negative Height,0
3,Negative Width,0


In [26]:
# ======================================================
# Freight Validation
# ======================================================

items = cleaned_data["olist_order_items_dataset"]

freight_validation = pd.DataFrame({

    "Check":[

        "Negative Freight",

        "Negative Price",

        "Zero Price"

    ],

    "Violations":[

        (items["freight_value"] < 0).sum(),

        (items["price"] < 0).sum(),

        (items["price"] == 0).sum()

    ]

})

display(freight_validation)

,Check,Violations
0,Negative Freight,0
1,Negative Price,0
2,Zero Price,0


In [27]:
# ======================================================
# Business Rule Dashboard
# ======================================================

total_violations = (

    rule_df["Violations"].sum()

    +

    payment_validation["Violations"].sum()

    +

    product_validation["Violations"].sum()

    +

    freight_validation["Violations"].sum()

)

print("="*70)

print("BUSINESS RULE VALIDATION")

print("="*70)

print(f"Total Violations : {total_violations:,}")

print("="*70)

print("\nOrders")

display(rule_df)

print("\nPayments")

display(payment_validation)

print("\nProducts")

display(product_validation)

print("\nOrder Items")

display(freight_validation)

BUSINESS RULE VALIDATION
Total Violations : 9

Orders


,Dataset,Rule,Violations
0,olist_orders_dataset,Delivery after Purchase,0
1,olist_orders_dataset,Estimated Date after Purchase,0



Payments


,Check,Violations
0,Negative Payments,0
1,Zero Payments,9
2,Negative Installments,0



Products


,Check,Violations
0,Negative Weight,0
1,Negative Length,0
2,Negative Height,0
3,Negative Width,0



Order Items


,Check,Violations
0,Negative Freight,0
1,Negative Price,0
2,Zero Price,0


In [28]:
# ======================================================
# Cleaning Recommendations
# ======================================================

recommendations = []

if (payments["payment_value"] < 0).sum() > 0:

    recommendations.append(
        "Investigate negative payment values."
    )

if (items["price"] <= 0).sum() > 0:

    recommendations.append(
        "Review zero or negative product prices."
    )

if (products["product_weight_g"] < 0).sum() > 0:

    recommendations.append(
        "Review negative product weights."
    )

if invalid_delivery > 0:

    recommendations.append(
        "Investigate orders with delivery dates before purchase dates."
    )

if len(recommendations) == 0:

    recommendations.append(

        "No business rule violations detected."

    )

print("="*70)

print("BUSINESS RECOMMENDATIONS")

print("="*70)

for i, rec in enumerate(recommendations, start=1):

    print(f"{i}. {rec}")

BUSINESS RECOMMENDATIONS
1. No business rule violations detected.


In [29]:
# ======================================================
# Cleaning Quality Score
# ======================================================

checks = (
    len(rule_df)
    + len(payment_validation)
    + len(product_validation)
    + len(freight_validation)
)

score = round(

    (

        1 -

        (total_violations / max(checks, 1))

    ) * 100,

    2

)

score = max(0, score)

print("="*70)

print("DATA CLEANING QUALITY SCORE")

print("="*70)

print(f"Validation Checks : {checks}")

print(f"Violations        : {total_violations}")

print(f"Quality Score     : {score:.2f}%")

print("="*70)

DATA CLEANING QUALITY SCORE
Validation Checks : 12
Violations        : 9
Quality Score     : 25.00%


In [30]:
# ======================================================
# Before vs After Cleaning Comparison
# ======================================================

comparison = []

for dataset_name in datasets.keys():

    raw_rows = len(datasets[dataset_name])

    clean_rows = len(cleaned_data[dataset_name])

    comparison.append({

        "Dataset": dataset_name,

        "Raw Rows": raw_rows,

        "Cleaned Rows": clean_rows,

        "Rows Removed": raw_rows - clean_rows

    })

comparison_df = pd.DataFrame(comparison)

display(comparison_df)

,Dataset,Raw Rows,Cleaned Rows,Rows Removed
0,olist_customers_dataset,99441,99441,0
1,olist_geolocation_dataset,1000163,738332,261831
2,olist_order_items_dataset,112650,112650,0
3,olist_order_payments_dataset,103886,103886,0
4,olist_order_reviews_dataset,99224,99224,0
5,olist_orders_dataset,99441,99441,0
6,olist_products_dataset,32951,32951,0
7,olist_sellers_dataset,3095,3095,0
8,product_category_name_translation,71,71,0


In [31]:
# ======================================================
# Cleaning Metrics Dashboard
# ======================================================

print("="*70)

print("DATA CLEANING METRICS")

print("="*70)

print(f"Datasets Processed : {len(cleaned_data)}")

print(f"Original Rows      : {comparison_df['Raw Rows'].sum():,}")

print(f"Cleaned Rows       : {comparison_df['Cleaned Rows'].sum():,}")

print(f"Rows Removed       : {comparison_df['Rows Removed'].sum():,}")

print("="*70)

DATA CLEANING METRICS
Datasets Processed : 9
Original Rows      : 1,550,922
Cleaned Rows       : 1,289,091
Rows Removed       : 261,831


In [32]:
# ======================================================
# Clean Dataset Registry
# ======================================================

print("="*70)
print("CLEAN DATASET REGISTRY")
print("="*70)

registry = []

for dataset_name, df in cleaned_data.items():

    registry.append({

        "Dataset": dataset_name,

        "Rows": len(df),

        "Columns": len(df.columns),

        "Memory (MB)": round(
            df.memory_usage(deep=True).sum()/1024**2,
            2
        )

    })

registry_df = pd.DataFrame(registry)

display(registry_df)

print("="*70)
print("All cleaned datasets are available in memory.")
print("Export step intentionally skipped.")
print("="*70)

CLEAN DATASET REGISTRY


,Dataset,Rows,Columns,Memory (MB)
0,olist_customers_dataset,99441,5,11.03
1,olist_geolocation_dataset,738332,5,42.81
2,olist_order_items_dataset,112650,7,16.33
3,olist_order_payments_dataset,103886,5,8.11
4,olist_order_reviews_dataset,99224,7,14.25
5,olist_orders_dataset,99441,8,14.80
6,olist_products_dataset,32951,9,3.95
7,olist_sellers_dataset,3095,4,0.22
8,product_category_name_translation,71,2,0.00


All cleaned datasets are available in memory.
Export step intentionally skipped.


In [33]:
# ======================================================
# Cleaning Audit Log
# ======================================================

audit_log = comparison_df.copy()

audit_log["Cleaning Date"] = pd.Timestamp.now()

audit_log["Notebook"] = "Notebook_02_Data_Cleaning"

display(audit_log)

,Dataset,Raw Rows,Cleaned Rows,Rows Removed,Cleaning Date,Notebook
0,olist_customers_dataset,99441,99441,0,2026-07-04 00:49:15.740402,Notebook_02_Data_Cleaning
1,olist_geolocation_dataset,1000163,738332,261831,2026-07-04 00:49:15.740402,Notebook_02_Data_Cleaning
2,olist_order_items_dataset,112650,112650,0,2026-07-04 00:49:15.740402,Notebook_02_Data_Cleaning
3,olist_order_payments_dataset,103886,103886,0,2026-07-04 00:49:15.740402,Notebook_02_Data_Cleaning
4,olist_order_reviews_dataset,99224,99224,0,2026-07-04 00:49:15.740402,Notebook_02_Data_Cleaning
5,olist_orders_dataset,99441,99441,0,2026-07-04 00:49:15.740402,Notebook_02_Data_Cleaning
6,olist_products_dataset,32951,32951,0,2026-07-04 00:49:15.740402,Notebook_02_Data_Cleaning
7,olist_sellers_dataset,3095,3095,0,2026-07-04 00:49:15.740402,Notebook_02_Data_Cleaning
8,product_category_name_translation,71,71,0,2026-07-04 00:49:15.740402,Notebook_02_Data_Cleaning


In [34]:
# ======================================================
# Final Validation
# ======================================================

validation = []

for dataset_name, df in cleaned_data.items():

    validation.append({

        "Dataset": dataset_name,

        "Missing Values": int(df.isnull().sum().sum()),

        "Duplicate Rows": int(df.duplicated().sum()),

        "Rows": len(df),

        "Columns": len(df.columns)

    })

validation_df = pd.DataFrame(validation)

display(validation_df)

,Dataset,Missing Values,Duplicate Rows,Rows,Columns
0,olist_customers_dataset,0,0,99441,5
1,olist_geolocation_dataset,0,0,738332,5
2,olist_order_items_dataset,0,0,112650,7
3,olist_order_payments_dataset,0,0,103886,5
4,olist_order_reviews_dataset,145903,0,99224,7
5,olist_orders_dataset,4908,0,99441,8
6,olist_products_dataset,2448,0,32951,9
7,olist_sellers_dataset,0,0,3095,4
8,product_category_name_translation,0,0,71,2


In [35]:
# ======================================================
# Notebook 02 Completion Report
# ======================================================

print("="*70)
print("RETAIL INTELLIGENCE PLATFORM")
print("NOTEBOOK 02 - DATA CLEANING")
print("="*70)

modules = [

    "Dataset Loading",

    "Working Copy Creation",

    "Data Type Conversion",

    "Missing Value Assessment",

    "Duplicate Cleaning",

    "Business Rule Validation",

    "Cleaning Quality Assessment",

    "Before vs After Comparison",

    "Clean Dataset Export",

    "Cleaning Audit Log"

]

print("\nCompleted Modules\n")

for i, module in enumerate(modules, start=1):

    print(f"{i:02d}. {module}")

print("\nStatus : COMPLETED")

print("\nNext Notebook")

print("Notebook 03 - Exploratory Data Analysis")

print("="*70)

RETAIL INTELLIGENCE PLATFORM
NOTEBOOK 02 - DATA CLEANING

Completed Modules

01. Dataset Loading
02. Working Copy Creation
03. Data Type Conversion
04. Missing Value Assessment
05. Duplicate Cleaning
06. Business Rule Validation
07. Cleaning Quality Assessment
08. Before vs After Comparison
09. Clean Dataset Export
10. Cleaning Audit Log

Status : COMPLETED

Next Notebook
Notebook 03 - Exploratory Data Analysis
